In [ ]:
import sys
import os
import random
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time

import torch
import torch.nn as nn
from torch_geometric.data import Dataset as PyGDataset, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.utils import subgraph, to_dense_batch
from torch_scatter import scatter_add, scatter_mean

warnings.filterwarnings("ignore", category=FutureWarning, module="torch")
torch.backends.cudnn.benchmark = True

# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================
CONFIG = {
    # --- Experiment Setup ---
    "SEED": 0,
    "DEVICE": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # --- Data Paths (Adjust these relative paths for Github) ---
    "DATA_DIR": Path("../data/splits/"),  # Assumed path for processed CSVs
    "PROTEIN_GRAPH_DIR": Path("../Data/Protein_Graphs_PyG/"),
    "LIGAND_GRAPH_DIR": Path("../Data/Ligand_Graphs_PyG/"),
    "GPCR_INFO_FILE": Path("../data/resources/ChEMBL_GPCR_Info.csv"),
    
    # --- Model Checkpoint ---
    # Ensure the user knows to place their trained model here
    "MODEL_SAVE_PATH": Path("../models/saved/gpcract_final.pt"),
    "CACHE_DIR": Path("./cache_attention_analysis/"),

    # --- Model Hyperparameters (Must match training config) ---
    "HIDDEN_DIM": 128,
    "PROTEIN_LAYERS": 4,
    "PROPAGATION_ATTENTION_LAYERS": 3,
    "ATTENTION_HEADS": 4,
    "PROTEIN_TYPE": "gated_residual",
    "LIGAND_TYPE": "gated_residual",
    "ELEMENT_EMBEDDING_DIM": 8,
    "DROPOUT": 0.0, # Dropout is irrelevant for inference/analysis
    
    "GPU_BATCH_SIZE": 8 # Smaller batch size for attention extraction to avoid OOM
}

In [ ]:
# ==============================================================================
# 1. MODEL DEFINITIONS
# ==============================================================================

def unsorted_segment_sum(data, segment_ids, num_segments):
    out = data.new_zeros((num_segments, data.size(1)))
    scatter_add(data, segment_ids, out=out, dim=0)
    return out

class E_GCL_Gated(nn.Module):
    """E(n) Equivariant Graph Convolutional Layer with Gating"""
    def __init__(self, input_nf, output_nf, hidden_nf, edges_in_d=0, act_fn=nn.SiLU(), residual=True, attention=False, normalize=False, coords_agg='mean', tanh=False):
        super(E_GCL_Gated, self).__init__()
        input_edge = input_nf * 2
        self.residual = residual
        self.attention = attention
        self.normalize = normalize
        self.coords_agg = coords_agg
        self.tanh = tanh
        self.epsilon = 1e-8
        edge_coords_nf = 1

        self.edge_mlp = nn.Sequential(
            nn.Linear(input_edge + edge_coords_nf + edges_in_d, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, hidden_nf),
            act_fn)
        
        self.node_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf))
        
        self.gate_mlp = nn.Sequential(
            nn.Linear(hidden_nf + input_nf, hidden_nf),
            act_fn,
            nn.Linear(hidden_nf, output_nf),
            nn.Sigmoid())

        self.node_norm = nn.LayerNorm(output_nf)
        
        layer = nn.Linear(hidden_nf, 1, bias=False)
        torch.nn.init.xavier_uniform_(layer.weight, gain=0.001)
        coord_mlp_list = [nn.Linear(hidden_nf, hidden_nf), act_fn, layer]
        if self.tanh: coord_mlp_list.append(nn.Tanh())
        self.coord_mlp = nn.Sequential(*coord_mlp_list)

        if self.attention:
            self.att_mlp = nn.Sequential(nn.Linear(hidden_nf, 1), nn.Sigmoid())

    def coord2radial(self, edge_index, coord):
        row, col = edge_index
        coord_diff = coord[row] - coord[col]
        radial = torch.sum(coord_diff**2, dim=1, keepdim=True)
        if self.normalize:
            norm = torch.sqrt(radial + self.epsilon)
            coord_diff = coord_diff / norm
        return radial, coord_diff

    def edge_model(self, h_row, h_col, radial, edge_attr):
        out = torch.cat([h_row, h_col, radial] + ([edge_attr] if edge_attr is not None else []), dim=1)
        out = self.edge_mlp(out)
        if self.attention:
            out = out * self.att_mlp(out)
        return out

    def node_model(self, x, edge_index, edge_feat, node_attr=None):
        row, col = edge_index
        agg = unsorted_segment_sum(edge_feat, row, num_segments=x.size(0))
        agg_cat = torch.cat([x, agg] + ([node_attr] if node_attr is not None else []), dim=1)
        
        update_val = self.node_mlp(agg_cat)
        gate_val = self.gate_mlp(agg_cat)
        
        out = x * gate_val + (x + update_val) * (1 - gate_val) if self.residual else x * gate_val + update_val * (1 - gate_val)
        return self.node_norm(out)

    def coord_model(self, coord, edge_index, coord_diff, edge_feat):
        row, col = edge_index
        trans = coord_diff * self.coord_mlp(edge_feat)
        if self.coords_agg == 'sum':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0))
        elif self.coords_agg == 'mean':
            agg = unsorted_segment_sum(trans, row, num_segments=coord.size(0)) / (unsorted_segment_sum(torch.ones_like(trans), row, num_segments=coord.size(0)) + 1e-8)
        else:
            raise Exception('Wrong coords_agg parameter')
        return coord + agg

    def forward(self, h, edge_index, coord, edge_attr=None, node_attr=None):
        row, col = edge_index
        radial, coord_diff = self.coord2radial(edge_index, coord)
        e_ij = self.edge_model(h[row], h[col], radial, edge_attr)
        coord = self.coord_model(coord, edge_index, coord_diff, e_ij)
        h = self.node_model(h, edge_index, e_ij, node_attr)
        return h, coord, e_ij

class EGNN_Gated_GlobalResidual(nn.Module):
    def __init__(self, in_node_nf, hidden_nf, out_node_nf, in_edge_nf=0, n_layers=4, residual=True, attention=False, normalize=False, coords_agg='mean', tanh=False):
        super().__init__()
        self.hidden_nf = hidden_nf
        self.n_layers = n_layers
        self.embedding_in = nn.Linear(in_node_nf, hidden_nf)
        self.embedding_out = nn.Linear(hidden_nf, out_node_nf)
        for i in range(n_layers):
            self.add_module(f"gcl_{i}", E_GCL_Gated(hidden_nf, hidden_nf, hidden_nf, edges_in_d=in_edge_nf, residual=residual, attention=attention, normalize=normalize, coords_agg=coords_agg, tanh=tanh))

    def forward(self, h, coord, edge_index, edge_attr=None):
        device = self.embedding_in.weight.device
        h, coord, edge_index = h.to(device), coord.to(device), edge_index.to(device)
        if edge_attr is not None: edge_attr = edge_attr.to(device)
        
        h_initial = self.embedding_in(h)
        h = h_initial
        for i in range(self.n_layers):
            h, coord, _ = self._modules[f"gcl_{i}"](h, edge_index, coord, edge_attr=edge_attr)
        h = h + h_initial
        return self.embedding_out(h), coord

def create_encoder(config, in_dim, hidden_dim):
    # Simplified to only support Gated Residual as used in GPCRact
    return EGNN_Gated_GlobalResidual(
        in_node_nf=in_dim, hidden_nf=hidden_dim, out_node_nf=hidden_dim,
        n_layers=config['n_layers'], attention=True, tanh=True
    )

class GPCRact_Model(nn.Module):
    """
    GPCRact Model Architecture (Cleaned for Inference/Analysis)
    Removed Class/Family embeddings.
    """
    def __init__(self, protein_in_dim_clean, protein_in_dim_full, ligand_in_dim, hidden_dim,
                 protein_config, ligand_config, element_embedding_dim,
                 n_attn_heads, dropout, propagation_attention_layers):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # --- Module 1: Interaction ---
        self.element_embedding = nn.Embedding(num_embeddings=6, embedding_dim=element_embedding_dim)
        self.bs_encoder = create_encoder(protein_config, protein_in_dim_clean, hidden_dim)
        self.ligand_encoder = create_encoder(ligand_config, ligand_in_dim, hidden_dim)
        self.p_to_l_attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=n_attn_heads, dropout=dropout, batch_first=True)

        # --- Module 2: Local Propagation (EGNN) ---
        propagation_config = {"type": protein_config['type'], "n_layers": protein_config['n_layers']}
        self.local_propagation_encoder = create_encoder(propagation_config, hidden_dim, hidden_dim)

        self.protein_embedding_ca = nn.Linear(protein_in_dim_full, hidden_dim)
        self.protein_embedding_sc = nn.Linear(protein_in_dim_full, hidden_dim)

        # --- Module 3: Global Propagation (Transformer) ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_attn_heads, dim_feedforward=hidden_dim * 4,
            dropout=dropout, batch_first=True
        )
        self.global_integration_transformer = nn.TransformerEncoder(encoder_layer, num_layers=propagation_attention_layers)

        # --- Prediction Head ---
        self.activity_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, protein_batch, ligand_batch):
        # 1. Binding Interaction (Cross-Attention)
        bs_mask = protein_batch.bs_mask
        bs_edge_index, _ = subgraph(bs_mask, protein_batch.edge_index, relabel_nodes=True, num_nodes=protein_batch.num_nodes)
        bs_pos = protein_batch.pos[bs_mask]

        p_features_bs_clean = torch.cat([
            protein_batch.x_float_clean[bs_mask],
            self.element_embedding(protein_batch.x_elem[bs_mask])
        ], dim=1)
        
        h_p_bs, _ = self.bs_encoder(p_features_bs_clean, bs_pos, bs_edge_index)
        h_l, _ = self.ligand_encoder(ligand_batch.x, ligand_batch.pos, ligand_batch.edge_index)

        padded_bs_h, bs_padding_mask = to_dense_batch(h_p_bs, protein_batch.batch[bs_mask])
        padded_l_h, l_padding_mask = to_dense_batch(h_l, ligand_batch.batch)

        p_updated_padded_h, _ = self.p_to_l_attention(query=padded_bs_h, key=padded_l_h, value=padded_l_h, key_padding_mask=~l_padding_mask)
        p_updated_padded_h[~bs_padding_mask] = 0
        ligand_signal = p_updated_padded_h[bs_padding_mask]

        # 2. Allosteric Propagation (Local EGNN)
        p_features_full = torch.cat([protein_batch.x_float_full, self.element_embedding(protein_batch.x_elem)], dim=1)
        h = torch.zeros(p_features_full.size(0), self.hidden_dim, device=p_features_full.device)
        
        ca_mask = (protein_batch.node_roles == 0)
        sc_mask = (protein_batch.node_roles == 1)
        h[ca_mask] = self.protein_embedding_ca(p_features_full[ca_mask]).to(h.dtype)
        h[sc_mask] = self.protein_embedding_sc(p_features_full[sc_mask]).to(h.dtype)

        coord = protein_batch.pos.clone()
        h_initial_for_residual = h.clone()

        for i in range(self.local_propagation_encoder.n_layers):
            h[bs_mask] = h[bs_mask] + ligand_signal
            h, _, _ = self.local_propagation_encoder._modules[f"gcl_{i}"](h, protein_batch.edge_index, coord)

        h = h + h_initial_for_residual
        h_after_egnn = self.local_propagation_encoder.embedding_out(h)

        # 3. Global Integration (Transformer)
        padded_h, padding_mask = to_dense_batch(h_after_egnn, protein_batch.batch)
        final_padded_h = self.global_integration_transformer(padded_h, src_key_padding_mask=~padding_mask)
        final_h_p = final_padded_h[padding_mask]

        # 4. Prediction
        pooled_activity_vector = scatter_mean(final_h_p, protein_batch.batch, dim=0)
        return self.activity_head(pooled_activity_vector)

In [ ]:
class GraphDataset(PyGDataset):
    def __init__(self, root, df, protein_graph_dir, ligand_graph_dir):
        self.df = df.reset_index(drop=True)
        self.protein_graph_dir = Path(protein_graph_dir)
        self.ligand_graph_dir = Path(ligand_graph_dir)
        super().__init__(root)

    def len(self):
        return len(self.df)

    def get(self, idx):
        row = self.df.iloc[idx]
        ikey, uniprot_id = row['Ikey'], row['AC']
        
        try:
            protein_graph = torch.load(self.protein_graph_dir / f"{uniprot_id}.pt", map_location='cpu')
            ligand_graph = torch.load(self.ligand_graph_dir / f"{ikey}.pt", map_location='cpu')
            
            original_x = protein_graph.x
            
            # --- Graph Feature Slicing ---
            h_res_type = original_x[:, :20]
            h_is_bs    = original_x[:, 20:21]
            h_disp     = original_x[:, 21:23]
            protein_graph.x_elem = original_x[:, 23].long()
            h_rel_pos  = original_x[:, 24:27]
            h_dist_ca  = original_x[:, 27:28]
            h_rdkit    = original_x[:, 28:]

            protein_graph.x_float_full = torch.cat([h_res_type, h_is_bs, h_disp, h_rel_pos, h_dist_ca, h_rdkit], dim=1)
            protein_graph.x_float_clean = torch.cat([h_res_type, h_rel_pos, h_dist_ca, h_rdkit], dim=1)
            
            del protein_graph.x
            protein_graph.node_roles = protein_graph.node_role
            del protein_graph.node_role

            protein_graph.activity_label = torch.tensor([float(row['Label'])], dtype=torch.float)
            protein_graph.ikey = ikey
            protein_graph.uniprot_id = uniprot_id

            return protein_graph, ligand_graph

        except FileNotFoundError:
            return None, None

def collate_fn(data_list):
    valid_data = [item for item in data_list if item[0] is not None]
    if not valid_data: return None
    p, l = zip(*valid_data)
    return Batch.from_data_list(p), Batch.from_data_list(l)

In [ ]:
class CustomTransformerEncoderLayer(nn.TransformerEncoderLayer):
    """
    A custom TransformerEncoderLayer to extract attention weights during forward pass.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def forward(self, src, src_mask=None, src_key_padding_mask=None, is_causal=False):
        x = src
        if self.norm_first:
            sa_out, _ = self._sa_block(self.norm1(x), src_mask, src_key_padding_mask, is_causal=is_causal)
            x = x + sa_out
            x = x + self._ff_block(self.norm2(x))
        else:
            sa_out, _ = self._sa_block(x, src_mask, src_key_padding_mask, is_causal=is_causal)
            x = self.norm1(x + sa_out)
            x = self.norm2(x + self._ff_block(x))
        return x

    def _sa_block(self, x, attn_mask, key_padding_mask, is_causal=False):
        # average_attn_weights=False returns (batch, heads, target_len, source_len)
        x, attn_weights = self.self_attn(x, x, x,
                                           attn_mask=attn_mask,
                                           key_padding_mask=key_padding_mask,
                                           need_weights=True,
                                           average_attn_weights=False, 
                                           is_causal=is_causal)
        return self.dropout1(x), attn_weights

In [ ]:
def extract_attention_scores(config):
    """
    Loads model, processes test set, extracts attention weights, and averages across heads.
    """
    print("--- Initializing Model and Data for Analysis ---")
    device = config["DEVICE"]
    config["RESULTS_DIR"].mkdir(parents=True, exist_ok=True)
    config["CACHE_DIR"].mkdir(parents=True, exist_ok=True)

    # 1. Prepare Data
    test_df = pd.read_csv(config["DATA_DIR"] / "test_set_scaf.csv")
    # Filter for existing files
    p_exists = (config['PROTEIN_GRAPH_DIR'] / (test_df['AC'] + '.pt')).apply(os.path.exists)
    l_exists = (config['LIGAND_GRAPH_DIR'] / (test_df['Ikey'] + '.pt')).apply(os.path.exists)
    test_df = test_df[p_exists & l_exists].reset_index(drop=True)
    
    test_dataset = GraphDataset(
        root=str(config['CACHE_DIR']), 
        df=test_df,
        protein_graph_dir=config['PROTEIN_GRAPH_DIR'], 
        ligand_graph_dir=config['LIGAND_GRAPH_DIR']
    )
    test_loader = DataLoader(test_dataset, batch_size=config['GPU_BATCH_SIZE'], shuffle=False, collate_fn=collate_fn, num_workers=4)
    
    # 2. Initialize Model
    print("Initializing model...")
    p_sample, l_sample = next(item for item in test_dataset if item[0] is not None)
    
    model = GPCRact_Model(
        protein_in_dim_clean=p_sample.x_float_clean.shape[1] + config['ELEMENT_EMBEDDING_DIM'],
        protein_in_dim_full=p_sample.x_float_full.shape[1] + config['ELEMENT_EMBEDDING_DIM'],
        ligand_in_dim=l_sample.x.shape[1],
        hidden_dim=config['HIDDEN_DIM'],
        protein_config={"type": config['PROTEIN_TYPE'], "n_layers": config['PROTEIN_LAYERS']},
        ligand_config={"type": config['LIGAND_TYPE'], "n_layers": config['LIGAND_LAYERS']},
        element_embedding_dim=config['ELEMENT_EMBEDDING_DIM'],
        dropout=config['DROPOUT'],
        n_attn_heads=config['ATTENTION_HEADS'],
        propagation_attention_layers=config['PROPAGATION_ATTENTION_LAYERS']
    )

    print(f"Loading trained weights from {config['MODEL_SAVE_PATH']}")
    # Ensure strict=False in case of minor mismatches (e.g., removed embedding layers)
    state_dict = torch.load(config['MODEL_SAVE_PATH'], map_location=device)
    # Filter out class/family embeddings if present in checkpoint but not in model
    state_dict = {k: v for k, v in state_dict.items() if "gpcr_class_embedding" not in k and "gpcr_family_embedding" not in k}
    model.load_state_dict(state_dict, strict=False)
    model.to(device)
    
    # 3. Inject Custom Transformer Layer for Hooks
    print("Injecting custom transformer layers for attention extraction...")
    for i in range(len(model.global_integration_transformer.layers)):
        old_layer = model.global_integration_transformer.layers[i]
        new_layer = CustomTransformerEncoderLayer(
            d_model=config['HIDDEN_DIM'], nhead=config['ATTENTION_HEADS'],
            dim_feedforward=config['HIDDEN_DIM'] * 4, dropout=config['DROPOUT'],
            batch_first=True
        )
        new_layer.load_state_dict(old_layer.state_dict())
        new_layer.to(device)
        model.global_integration_transformer.layers[i] = new_layer
    model.eval()

    # 4. Register Hook
    attention_weights_store = {}
    def hook_fn(module, input, output):
        # output[1] is weights: [Batch, Heads, Target_Len, Source_Len]
        attention_weights_store['current_batch'] = output[1].detach().cpu()

    target_module = model.global_integration_transformer.layers[-1].self_attn
    hook_handle = target_module.register_forward_hook(hook_fn)
    
    # 5. Extract and Average Attention
    atomic_attention_points = []
    print("Extracting attention scores...")
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Processing Batches"):
            if batch is None: continue
            protein_batch, ligand_batch = batch
            protein_batch, ligand_batch = protein_batch.to(device), ligand_batch.to(device)
            
            model(protein_batch, ligand_batch)
            batch_attn = attention_weights_store.get('current_batch') # [B, Heads, N, N]
            if batch_attn is None: continue

            # Average across heads [B, N, N]
            batch_attn_mean = batch_attn.mean(dim=1) 

            graphs = protein_batch.to_data_list()
            for i, graph in enumerate(graphs):
                if not hasattr(graph, 'node_res_num'): continue # Ensure we have residue info

                # Identify Binding Site (Target) and Allosteric Site (Source) nodes
                bs_indices = torch.where(graph.bs_mask)[0].cpu()
                as_indices = torch.where(~graph.bs_mask)[0].cpu()

                if len(as_indices) == 0 or len(bs_indices) == 0: continue
                
                # Extract attention sub-matrix: How much AS nodes attend to BS nodes
                # Shape: [Num_AS, Num_BS] -> Sum over BS to get total importance of each AS node
                attn_scores = batch_attn_mean[i, as_indices][:, bs_indices].sum(dim=1)
                
                res_nums = graph.node_res_num.cpu().numpy()
                for idx, score in zip(as_indices, attn_scores):
                    res_num = res_nums[idx.item()]
                    atomic_attention_points.append({
                        'uniprot_ac': graph.uniprot_id,
                        'uniprot_res_num': res_num,
                        'attention_score': score.item()
                    })

    hook_handle.remove()
    
    # 6. Aggregate
    print("Aggregating results...")
    df_attn = pd.DataFrame(atomic_attention_points)
    # Sum scores for same residue across different samples (if multiple ligands for same protein)
    df_final = df_attn.groupby(['uniprot_ac', 'uniprot_res_num'])['attention_score'].mean().reset_index()
    
    print(f"Extraction complete. Processed {len(df_final)} residues.")
    return df_final

# Run Extraction
df_attention = extract_attention_scores(CONFIG)

In [ ]:
# ==============================================================================
# PART 0: IMPORT LIBRARIES
# ==============================================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import requests
import json
import os
import time
from tqdm.auto import tqdm
from pathlib import Path

# ==============================================================================
# PART 1: HELPER FUNCTIONS FOR BW NUMBERING (USER-PROVIDED)
# ==============================================================================
BW_CACHE_FILE = Path("./bw_numbering_cache.json")

def load_bw_cache():
    """Loads the BW numbering cache from a JSON file."""
    if BW_CACHE_FILE.exists() and BW_CACHE_FILE.stat().st_size > 0:
        with open(BW_CACHE_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_bw_cache(cache):
    """Saves the BW numbering cache to a JSON file."""
    with open(BW_CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=4)

def get_entry_name_from_uniprot_api(uniprot_ac):
    """Fetches the UniProt Entry Name (e.g., ADRB2_HUMAN) from a UniProt Accession."""
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_ac}.json"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json().get('uniProtkbId')
    except requests.exceptions.RequestException:
        return None

def get_bw_map_from_api(entry_name):
    """Fetches the BW map for a given UniProt Entry Name from GPCRdb."""
    api_entry_name = entry_name.lower()
    url = f"https://gpcrdb.org/services/residues/{api_entry_name}/"
    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        residues = response.json()
        if residues:
            return {
                str(res['sequence_number']): res['display_generic_number']
                for res in residues if res.get('display_generic_number')
            }
    except requests.exceptions.RequestException:
        return None

# 1. Merge Metadata
print("Merging with GPCR Info...")
gpcr_info = pd.read_csv(CONFIG['GPCR_INFO_FILE'])
gpcr_info['uniprot_ac'] = gpcr_info['UniProt Accessions'].str.split(';').str[0]
merged_df = pd.merge(df_attention, gpcr_info[['uniprot_ac', 'Receptor Family']], on='uniprot_ac', how='left')
merged_df.dropna(subset=['Receptor Family'], inplace=True)   

# 2. APPLY BW NUMBERING
unique_proteins = merged_df['uniprot_ac'].unique()
bw_map_cache = load_bw_cache()
all_bw_maps = {}

print(f"\n--- Step 2: Fetching BW Numbering for {len(unique_proteins)} proteins ---")
for uniprot_id in tqdm(unique_proteins, desc=f"Fetching BW Maps (Head {head_idx})"):
    cached_entry = next((entry for entry, data in bw_map_cache.items() if data.get('uniprot_id') == uniprot_id), None)
    if cached_entry:
        all_bw_maps[uniprot_id] = bw_map_cache[cached_entry]
        continue

    entry_name = get_entry_name_from_uniprot_api(uniprot_id)
    if not entry_name: continue
    
    if entry_name in bw_map_cache:
        all_bw_maps[uniprot_id] = bw_map_cache[entry_name]
        continue

    bw_map = get_bw_map_from_api(entry_name)
    if bw_map:
        bw_map['uniprot_id'] = uniprot_id
        all_bw_maps[uniprot_id] = bw_map
        bw_map_cache[entry_name] = bw_map
        save_bw_cache(bw_map_cache)
    time.sleep(0.1)

# 3. Define Functional Regions
def apply_bw_map(row, all_maps):
    protein_map = all_maps.get(row['uniprot_ac'])
    if protein_map:
        return protein_map.get(str(row['uniprot_res_num']))
    return None

merged_df['BW_Number'] = merged_df.apply(lambda row: apply_bw_map(row, all_bw_maps), axis=1)
print("\n BW Numbering successfully applied.")

def get_functional_region(bw_number):
    """
    Correctly parses BW numbers in 'Helix.Position' or 'Helix.Position_in_Helix x Generic_Position' format.
    """
    if pd.isna(bw_number):
        return 'Flexible Loop/Terminus'
    
    try:
        bw_str = str(bw_number)
        
        # Take only the part before 'x' to handle formats like '3.50x50' -> '3.50'
        core_bw = bw_str.split('x')[0]
        
        # Now, parse the standard 'Helix.Position' format
        if '.' not in core_bw:
            # This handles non-standard but valid IDs like 'H8' or parsing errors
            return 'Other'
            
        helix_str, pos_str = core_bw.split('.')
        
        # This handles cases like '23.51' which are valid but outside TM1-8
        if len(helix_str) > 1 and not helix_str.startswith('12'):
            return 'Other'

        helix = int(helix_str)
        pos = int(pos_str)

        # Classification logic is the same
        if helix == 2 and pos == 50: return 'Sodium-binding Site'
        if helix == 3 and pos == 50: return 'DRY Motif'
        if (helix == 5 and pos == 50) or (helix == 3 and pos == 40) or (helix == 6 and pos == 44): return 'PIF Motif'
        if helix == 6 and pos in [40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]: return 'CWxP Motif'
        if helix == 7 and pos == 53: return 'NPxxY Motif'
        
        return 'Other'
        
    except (ValueError, IndexError):
        return 'Flexible Loop/Terminus' # Catch-all for any other weird formats

merged_df['Functional_Region'] = merged_df['BW_Number'].apply(get_functional_region)

# 4. Calculate Percentile Rank per Protein (Normalize)
merged_df['attention_rank_percentile'] = merged_df.groupby('uniprot_ac')['attention_score'].rank(pct=True)

# 5. Plotting (Averaged over all heads)
families_to_show = [
    'Acetylcholine receptors (muscarinic)', '5-Hydroxytryptamine receptors',
    'Dopamine receptors', 'Adenosine receptors', 'Adrenoceptors', 'Cannabinoid receptors'
]
regions_to_show = ['DRY Motif', 'PIF Motif', 'CWxP Motif', 'NPxxY Motif', 'Flexible Loop/Terminus']
family_map = {
    'Acetylcholine receptors (muscarinic)': 'Acetylcholine', 
    '5-Hydroxytryptamine receptors': 'Serotonin',
    'Dopamine receptors': 'Dopamine', 'Adenosine receptors': 'Adenosine',
    'Adrenoceptors': 'Adrenoceptors', 'Cannabinoid receptors': 'Cannabinoid'
}

plot_df = merged_df[merged_df['Receptor Family'].isin(families_to_show) & merged_df['Functional_Region'].isin(regions_to_show)].copy()
plot_df['Receptor Family'] = plot_df['Receptor Family'].map(family_map)

# Aggregate Mean Importance
agg_df = plot_df.groupby(['Receptor Family', 'Functional_Region'])['attention_rank_percentile'].mean().reset_index()

# Plot
fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(
    data=agg_df,
    x='Functional_Region', y='attention_rank_percentile', hue='Receptor Family',
    order=regions_to_show, palette='tab10', edgecolor='black', ax=ax
)
ax.set_title('Mean Self-Attention Importance of Functional Motifs (All Heads Averaged)', fontsize=18, fontweight='bold')
ax.set_xlabel('Functional Region', fontsize=14)
ax.set_ylabel('Mean Importance (Percentile Rank)', fontsize=14)
ax.legend(title='Family', loc='upper right')
plt.tight_layout()
plt.show()
